<h3 style="color: #777777;"> Paso 0: Validar la ubicación de la base de datos</h3>

In [6]:
# MINIMAL DEBUG SCRIPT
import sqlite3
from pathlib import Path
import pandas as pd

print("1. Current directory:", Path.cwd())
print("\n2. Looking for nursery.db...")

# Try exact paths that should work
paths_to_try = [
    '../nursery.db',  # Most likely if you're in 'reports' folder
    'nursery.db',
    '../../nursery.db',
]

for path_str in paths_to_try:
    path = Path(path_str)
    print(f"\nTrying: {path_str}")
    print(f"  Absolute: {path.absolute()}")
    print(f"  Exists: {path.exists()}")
    
    if path.exists():
        try:
            conn = sqlite3.connect(str(path))
            cursor = conn.cursor()
            
            # List ALL tables
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
            tables = [t[0] for t in cursor.fetchall()]
            
            print(f"  ✅ Connected! Tables: {tables}")
            
            # If OBSERVACION_LOTE exists, show sample
            if 'OBSERVACION_LOTE' in tables:
                cursor.execute("SELECT * FROM OBSERVACION_LOTE LIMIT 3")
                sample = cursor.fetchall()
                print(f"  Sample from OBSERVACION_LOTE: {sample}")
            else:
                # Check for any observation-like table
                for table in tables:
                    if 'OBSERV' in table.upper():
                        cursor.execute(f"SELECT * FROM {table} LIMIT 3")
                        sample = cursor.fetchall()
                        print(f"  Sample from {table}: {sample}")
            
            conn.close()
            break
            
        except Exception as e:
            print(f"  ❌ Error: {e}")

1. Current directory: /Users/estefania/Documents/Data/my_portfolio_projects/scripts/nursery_manager

2. Looking for nursery.db...

Trying: ../nursery.db
  Absolute: /Users/estefania/Documents/Data/my_portfolio_projects/scripts/nursery_manager/../nursery.db
  Exists: False

Trying: nursery.db
  Absolute: /Users/estefania/Documents/Data/my_portfolio_projects/scripts/nursery_manager/nursery.db
  Exists: True
  ✅ Connected! Tables: ['sqlite_sequence', 'GASTOS', 'REGISTRO_CLIMA', 'ENTRADA_PLANTAS', 'ETAPA_CRECIMIENTO', 'OBSERVACION_LOTE', 'REGISTRO_RIEGO', 'METODOS_PROPAGACION', 'CONTROL_PLAGAS', 'OBSERVACION_LOTE_BACKUP_12_19_25', 'ESPECIE_LOTE', 'CATEGORIA']
  Sample from OBSERVACION_LOTE: [(1, 6, '2025-09-22', 76, 20.0, 'BOLSA_PEQUEÑA', 'Buena', 'Requerimiento control de plagas y división de 6 papayos'), (2, 4, '2025-09-23', 39, 15.0, 'BOLSA_PEQUEÑA', 'Buena', 'Tiene algunas hojas deformes'), (3, 3, '2025-09-24', 88, 33.0, 'PRE-EMBOLSE', 'Buena', 'Algunas hojas con huecos y bordes café y

<h3 style="color: #777777;"> Paso 1: Seleccionar los datos para el reporte</h3>

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import HTML, display

def get_observation_data(start_date=None, end_date=None, db_path='nursery.db'):
    """
    Read observation data from the OBSERVACION_LOTE table with optional date filtering.
    """
    # Convert to Path object and resolve relative path
    db_file = Path(db_path)
    
    # Check if database exists
    if not db_file.exists():
        raise FileNotFoundError(f"Database file not found: {db_file.absolute()}")
    
    try:
        # Connect to database 
        conn = sqlite3.connect(str(db_file))

        # Build the query with optional date filtering
        query = """ 
        SELECT * FROM OBSERVACION_LOTE_BACKUP_12_19_25
        WHERE 1=1 -- Always true makes it easier to add conditions
        """

        # Add date filters if provided
        params = []
        if start_date:
            query += " AND FECHA_OBSERVACION >= ?"
            params.append(start_date)
        if end_date:
            query += " AND FECHA_OBSERVACION <= ?"
            params.append(end_date)

        # Order by date (most recent first)
        query += " ORDER BY FECHA_OBSERVACION DESC"

        # Execute the query and get data
        df = pd.read_sql_query(query, conn, params=params if params else None)
        
        return df
    
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        raise
    finally:
        if 'conn' in locals():
            conn.close()

def generate_html_report(df, title="VIVERO RAÍCES DORADAS - REPORTE OPERATIVO"):
    """
    Generate an HTML report from observation data.
    
    Args:
        df (pandas.DataFrame): Observation data
        title (str): Report title
    
    Returns:
        str: HTML string
    """
    if df.empty:
        return "<h3>No data available</h3>"
    
    # Calculate statistics
    total_observations = len(df)
    date_range = f"{df['FECHA_OBSERVACION'].min()} a {df['FECHA_OBSERVACION'].max()}"
    unique_lots = df['ESPECIE_LOTE_ID'].nunique()
    available_columns = len(df.columns)
    
    # Create the HTML
    html_content = f'''
    <center><h1 style="color: #777777;">{title}</h1></center>
    <center><h2 style="color: #777777;">Observación de Plantas</h2></center>
    
    <div align="center">
        <table style="width: 80%; border-collapse: collapse; margin: 20px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Rango de Fechas</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Conteo de Observaciones</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Lotes Existentes</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Columnas disponibles</th>
            </tr>
            <tr>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{date_range}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{total_observations:,}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{unique_lots}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{available_columns}</td>
            </tr>
        </table>
    </div>
    
    <div align="center" style="margin: 20px;">
        <h3 style="color: #777777;">Resumen histórico de datos</h3>
        <table style="width: 60%; border-collapse: collapse; margin: 10px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Nombre de Columna</th>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Valores Únicos</th>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Valores Totales</th>
            </tr>
    '''
    
    # Add row for each column
    for column in df.columns:
        unique_count = df[column].nunique()
        value_count = df[column].shape[0]
        
        html_content += f'''
            <tr>
                <td style="padding: 8px; border: 1px solid #ddd; color: #333;">{column}</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{unique_count}</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{value_count}</td>
            </tr>
        '''
    
    # Close the column detail table
    html_content += '''
        </table>
    </div>
    
    <div align="center" style="margin: 30px;">
    <center><h2 style="color: #777777;">Análisis de Inventario</h2></center>
        <h3 style="color: #777777;">Conteo por Etapa de Crecimiento</h3>
        <table style="width: 60%; border-collapse: collapse; margin: 10px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d;">Etapa de Crecimiento</th>
                <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d;">Inventario Actual</th>
                <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d;">Porcentaje</th>
                
            </tr>
    '''
    
    # Add growth stage summary
    if 'ETAPA_CRECIMIENTO' in df.columns and 'CANTIDAD_PLANTAS_VIVAS' in df.columns:
        # Get the latest date in the dataset
        latest_date = df['FECHA_OBSERVACION'].max()
        
        # Get current inventory (most recent observation per lot)
        # Sort by date descending, then get first record per lot
        df_sorted = df.sort_values('FECHA_OBSERVACION', ascending=False)
        current_inventory = df_sorted.drop_duplicates(subset='ESPECIE_LOTE_ID', keep='first')
        
        # Calculate current plant count by stage (sum of CANTIDAD_PLANTAS_VIVAS)
        current_plant_counts = current_inventory.groupby('ETAPA_CRECIMIENTO')['CANTIDAD_PLANTAS_VIVAS'].sum()
        
        # Get total plants for percentage calculation
        total_current_plants = current_inventory['CANTIDAD_PLANTAS_VIVAS'].sum()
        # Sort descending plant count for readability
        current_plant_counts = current_plant_counts.sort_values(ascending=False)

        
        for stage, plant_count in current_plant_counts.items():
            # Calculate percentage based on current inventory
            percentage = (plant_count / total_current_plants) * 100 if total_current_plants > 0 else 0
            
            
            html_content += f'''
                <tr>
                    <td style="padding: 8px; border: 1px solid #ddd; color: #333;">{stage}</td>
                    <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">{plant_count:,}</td>
                    <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{percentage:.1f}%</td>
                </tr>
            '''
        
        # Add a summary row
        html_content += f'''
            <tr style="background-color: #f5f5f5;">
                <td style="padding: 10px; border: 1px solid #ddd; color: #333; font-weight: bold;">TOTAL</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">{total_current_plants:,}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #666; font-weight: bold;">100.0%</td>
            </tr>
        '''
        
        # Add a note about the current inventory date
        html_content += f'''
        </table>
        <p style="color: #777777; font-size: 12px; margin-top: 10px;">
            <em>Inventario actual basado en la última observación (Fecha más reciente: {latest_date})</em>
        </p>
    '''
    else:
        html_content += '''
            <tr>
                <td colspan="4" style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #666;">
                    No hay datos de etapa de crecimiento o cantidad de plantas disponibles
                </td>
            </tr>
        </table>
    '''
    
    # Add date for footer
    from datetime import datetime
    current_date = datetime.now().strftime('%Y-%m-%d')
    
    html_content += f'''
    </div>
    
    <div align="center" style="margin-top: 40px; padding: 20px; background-color: #f9f9f9; border-top: 2px solid #e3d0b3;">
        <p style="color: #777777; font-size: 12px;">
            Reporte generado automáticamente • Vivero Raíces Doradas • {current_date}
        </p>
    </div>
    '''
    
    return html_content


def display_report(df=None, start_date=None, end_date=None, db_path='nursery.db'):
    """
    Complete function to get data and display HTML report.
    
    Args:
        df (pandas.DataFrame, optional): Pre-loaded data. If None, loads from database.
        start_date (str, optional): Start date for filtering
        end_date (str, optional): End date for filtering
        db_path (str, optional): Database path
    """
    from datetime import datetime
    
    # Get data if not provided
    if df is None:
        print("📊 Cargando datos de la base de datos...")
        df = get_observation_data(start_date=start_date, end_date=end_date, db_path=db_path)
        print(f"✅ Datos cargados: {len(df)} registros")
    
    # Generate and display HTML report
    html_report = generate_html_report(df)
    display(HTML(html_report) )
    
    # Return the dataframe for further analysis
    return df

# Create the current_inventory dataframe:
def get_current_inventory(df):
    """
    Extract current inventory from observation data.
    Returns only the most recent observation per ESPECIE_LOTE_ID.
    """
    # Sort by date descending and keep first (most recent) per lot
    df_sorted = df.sort_values('FECHA_OBSERVACION', ascending=False)
    current_inventory = df_sorted.drop_duplicates(subset='ESPECIE_LOTE_ID', keep='first')
    
    return current_inventory


# Usage workflow
if __name__ == "__main__":
    #print("Iniciando reporte del vivero...\n")
    
    # Step 1: Display observation data
    print("Generando reporte filtrado...")
    observations = display_report(start_date='2025-10-27')

    # Step 2: Extract current inventory
    current_inventory = get_current_inventory(observations)
    

Generando reporte filtrado...
📊 Cargando datos de la base de datos...
✅ Datos cargados: 825 registros


In [19]:
import sqlite3
import pandas as pd
from pathlib import Path
from typing import Optional, List, Dict, Any
from IPython.display import HTML, display

def get_left_table_data(
        db_path: str = 'nursery.db',
        table_name: str = 'ESPECIE_LOTE',
        columns: Optional[List[str]] = None,
        left_join_key: str = ' ID',
        rename_key_for_join: Optional[str] = None
) -> pd.DataFrame:
    """
    Read data from a dimension table for left join operations.
    Provides traceability and displays shape overview in jupiter.

    Args:
        db_path: Path to SQLite database
        table_name: Name of the left table
        columns: Specific columns to select (None=all)
        left_join_key: Primary key column name in the left table
        rename_key_for_join: Optional new name for the key column

    Returns:
        Dataframe with left table data ready for merging
    """

    print(f"📊Paso 1: Cargando tabla {table_name} para el reporte, con identificador único {left_join_key}")

    db_file = Path(db_path)

    if not db_file.exists():
        raise FileNotFoundError(f"Database file not found: {db_file.absolute()}")
    try:
        conn = sqlite3.connect(str(db_file))

        # Get table info for validation
        table_info = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)

        available_columns = table_info['name'].to_list()

        # Validate key column exists
        if left_join_key not in available_columns:
            raise ValueError(f"Columna identificador '{left_join_key}' se está en la tabla {table_name}")
        
        if columns:
            # Validate all requested columns exists
            missing_cols = [col for col in columns if col not in available_columns]

            if missing_cols:
                raise ValueError(f"Columnas NO disponibles en la tabla '{table_name}': {missing_cols}")
            
            column_str = ', '.join(columns)
            selected_columns = columns
        else:
            column_str = '*'
            selected_columns = available_columns

        # Build and execute query
        query = f"""
        SELECT {column_str}
        FROM {table_name}
        """

        df = pd.read_sql_query(query,conn)

        # Rename key column if specified
        if rename_key_for_join and rename_key_for_join != left_join_key:
            df = df.rename(columns={left_join_key: rename_key_for_join})
            print(f" ♻️ Columna identificadora renombrada de '{left_join_key}' to '{rename_key_for_join}'")

        # Display shape overview
        print(f"✅ TABLA CARGADA EXITOSAMENTE")
        print(f" Filas cargadas: {df.shape[0]}, Columnas cargadas: {df.shape[1]}")
        print(f" Columnas selecionadas: {', '.join(df.columns.tolist())} ")

        return df
    
    except sqlite3.Error as e:
        print(f"🔴 Database error: {e}")
        raise
    finally:
        if 'conn' in locals():
            conn.close()

# Load left table (species data)
especie_df = get_left_table_data(
    db_path='nursery.db',
    table_name='ESPECIE_LOTE',
    columns=['ID', 'NOMBRE_COMUN', 'NOMBRE_CIENTIFICO', 'FAMILIA', 'CATEGORIA'],
    left_join_key='ID',
    rename_key_for_join='ESPECIE_LOTE_ID'
)

📊Paso 1: Cargando tabla ESPECIE_LOTE para el reporte, con identificador único ID
 ♻️ Columna identificadora renombrada de 'ID' to 'ESPECIE_LOTE_ID'
✅ TABLA CARGADA EXITOSAMENTE
 Filas cargadas: 35, Columnas cargadas: 5
 Columnas selecionadas: ESPECIE_LOTE_ID, NOMBRE_COMUN, NOMBRE_CIENTIFICO, FAMILIA, CATEGORIA 


In [23]:
def enrich_with_left_table(
    right_df: pd.DataFrame,
    left_df: pd.DataFrame,
    right_join_key: str,
    left_join_key: str = None,
    columns_to_enrich: Optional[List[str]] = None,
    suffix: str = '_dim'
) -> pd.DataFrame:
    """ 
    Enrich right table data with left table information using left join.
    To be executed after loading left table data.

    Args:
        right_df: DataFrame with observation data
        left_df: DataFrame with data from get_left_table_data()
        right_join_key: Foreign key column name in right_df
        left_join_key: Key column name in left_df (if different)
        columns_to_enrich: Specific columns from left_df to include (None = all exept join key)
        sufix: Suffix to add to duplicate column names from left table

    Returs:
        Enriched DataFrame
    """
    print(f"🔗 Paso 2: Cruce de tablas creando un reporte unificado")
    #print(f"   Right table shape: {right_df.shape}")
    #print(f"   Left table shape: {left_df.shape}")
    #print(f"   Join key - Right: {right_join_key}")
    #print(f"   Join key - Left: {left_join_key if left_join_key else 'ID'}")
    
    # Use default left join key if not specified
    if left_join_key is None:
        left_join_key = right_join_key
    
    # Validate join keys exist
    if right_join_key not in right_df.columns:
        raise ValueError(f"Join key '{right_join_key}' not found in right table columns: {right_df.columns.tolist()}")
    
    if left_join_key not in left_df.columns:
        raise ValueError(f"Join key '{left_join_key}' not found in left table columns: {left_df.columns.tolist()}")
    
    # Prepare left table data for merge
    merge_df = left_df.copy()
    
    # Select specific columns to enrich
    if columns_to_enrich:
        # Always include the join key
        columns_to_select = [left_join_key] + columns_to_enrich
        merge_df = merge_df[columns_to_select]
    
    # Identify overlapping columns (excluding join keys)
    overlapping_cols = set(right_df.columns) & set(merge_df.columns) - {right_join_key, left_join_key}
    if overlapping_cols:
        print(f"   ⚠️  Warning: Overlapping column names: {list(overlapping_cols)}")
        print(f"   Will use suffix '{suffix}' for left table columns")
    
    # Perform left join
    #print(f"\n   Merging with parameters:")
    #print(f"   - how: 'left'")
    #print(f"   - left_on: {right_join_key}")
    #print(f"   - right_on: {left_join_key}")
    #print(f"   - suffixes: ('', '{suffix}')")
    
    enriched_df = pd.merge(
        right_df,
        merge_df,
        left_on=right_join_key,
        right_on=left_join_key,
        how='left',
        suffixes=('', suffix)
    )
    
    # Display results
    print(f"\n✅ CRUCE EXITOSO")
    print(f"   Tabla final: {enriched_df.shape[0]} filas × {enriched_df.shape[1]} columnas")
    print(f"   Incluidas {enriched_df.shape[1] - right_df.shape[1]} nuevas columnas")
    
    # Show merge summary
    matched_count = enriched_df[left_join_key].notna().sum()
    unmatched_count = len(enriched_df) - matched_count
    
    #print(f"\n📊 MERGE SUMMARY:")
    #print(f"   Total rows: {len(enriched_df)}")
    #print(f"   Matched rows: {matched_count} ({matched_count/len(enriched_df)*100:.1f}%)")
    #print(f"   Unmatched rows: {unmatched_count} ({unmatched_count/len(enriched_df)*100:.1f}%)")
    
    # Display preview of enriched data
    #print(f"\n🎯 ENRICHED DATA PREVIEW:")
    display(enriched_df.head().style.set_caption("First 5 rows of enriched data"))
    
    return enriched_df

enriched_inventory = enrich_with_left_table(
    right_df=current_inventory,
    left_df=especie_df,
    right_join_key='ESPECIE_LOTE_ID',
    left_join_key='ESPECIE_LOTE_ID',
    columns_to_enrich=['NOMBRE_COMUN', 'NOMBRE_CIENTIFICO', 'FAMILIA', 'CATEGORIA']
)


🔗 Paso 2: Cruce de tablas creando un reporte unificado

✅ CRUCE EXITOSO
   Tabla final: 34 filas × 12 columnas
   Incluidas 4 nuevas columnas


,ID,ESPECIE_LOTE_ID,FECHA_OBSERVACION,CANTIDAD_PLANTAS_VIVAS,ALTURA_PROMEDIO_CM,ETAPA_CRECIMIENTO,INDICADOR_SALUD,NOTAS,NOMBRE_COMUN,NOMBRE_CIENTIFICO,FAMILIA,CATEGORIA
0,971,1,2025-12-16,30,10.000000,PLANTULA_ALMACIGO,Buena,Sin Observaciones,Roble Rosado,Tabebuia rosea,Bignoniaceae,NATIVO
1,1010,27,2025-12-16,8,80.000000,BOLSA_GRANDE,Buena,Sin Observaciones,Lengua de Suegra Amarilla,Dracaena trifasciata Laurentii,Asparagaceae,ORNAMENTAL
2,999,20,2025-12-16,1,200.000000,BOLSA_GRANDE,Buena,Sin Observaciones,Maracuyá,Pasiflora edulis,Passifloraceae,FRUTAL
3,1000,21,2025-12-16,3,24.000000,BOLSA_PEQUEÑA,Buena,Sin Observaciones,Lulo,Solanum quitoense,Solanaceae,FRUTAL
4,1001,22,2025-12-16,6,8.000000,PLANTULA_ALMACIGO,Buena,Sin Observaciones,Icaco,Chrysobalanus icaco L,Chrysobalanaceae,NATIVO
